# Task 2: Food Outlet Descriptive Analysis

Dataset selected: **Zomato restaurant dataset (Kaggle source mirrored on GitHub raw CSV)**.

Objective: Perform descriptive analysis to identify customer preferences and business trends.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1) Load dataset using Pandas

In [ ]:
url = 'https://raw.githubusercontent.com/utkarsh-us/zomato-restaurant-analytics/master/dataset/zomato_data.csv'
df = pd.read_csv(url)

print('Shape:', df.shape)
df.head()

## 2) Data cleaning

In [ ]:
print('Missing values before cleaning:', int(df.isna().sum().sum()))
print('Duplicate rows before cleaning:', int(df.duplicated().sum()))

df = df.drop_duplicates().dropna().copy()
df['rate_num'] = pd.to_numeric(df['rate'].astype(str).str.replace('/5', '', regex=False), errors='coerce')
df['approx_cost(for two people)'] = pd.to_numeric(df['approx_cost(for two people)'], errors='coerce')
df = df.dropna(subset=['rate_num', 'approx_cost(for two people)']).copy()

print('Shape after cleaning:', df.shape)
df[['rate', 'rate_num', 'approx_cost(for two people)']].head()

## 3) Business questions and descriptive analysis

In [ ]:
# Q1: Which outlet type is most common?
type_counts = df['listed_in(type)'].value_counts()
print('Q1 - Most common outlet type:', type_counts.idxmax())
print(type_counts)

# Q2: What percentage of restaurants offer online orders?
online_pct = (df['online_order'].value_counts(normalize=True) * 100).round(2)
print('\nQ2 - Online order percentages (%):')
print(online_pct)

# Q3: What is the average customer rating overall and by type?
avg_rating = round(df['rate_num'].mean(), 2)
avg_rating_by_type = df.groupby('listed_in(type)')['rate_num'].mean().round(2).sort_values(ascending=False)
print('\nQ3 - Average rating overall:', avg_rating)
print('Average rating by outlet type:')
print(avg_rating_by_type)

# Q4: Which restaurants have the highest number of customer reviews (votes)?
top_reviewed = df[['name', 'votes', 'rate_num']].sort_values('votes', ascending=False).head(10)
print('\nQ4 - Top restaurants by votes:')
print(top_reviewed)

# Q5: Which price range is the most common?
price_bins = [0, 300, 600, 900, float('inf')]
price_labels = ['Low (<=300)', 'Medium (301-600)', 'High (601-900)', 'Premium (>900)']
df['price_range'] = pd.cut(df['approx_cost(for two people)'], bins=price_bins, labels=price_labels)
price_dist = df['price_range'].value_counts().sort_index()
print('\nQ5 - Most common price range:', price_dist.idxmax())
print(price_dist)

## 4) Charts

In [ ]:
# Chart 1: Outlet type distribution
plt.figure()
sns.countplot(data=df, x='listed_in(type)', order=type_counts.index, palette='Set2')
plt.title('Outlet Type Distribution')
plt.xlabel('Outlet Type')
plt.ylabel('Number of Restaurants')
plt.show()

In [ ]:
# Chart 2: Online order share
plt.figure()
online_pct.sort_values(ascending=False).plot(kind='bar', color=['#4c72b0', '#55a868'])
plt.title('Online Order Availability (%)')
plt.xlabel('Online Order')
plt.ylabel('Percentage')
plt.ylim(0, 100)
plt.show()

In [ ]:
# Chart 3: Cost distribution for two people
plt.figure()
sns.histplot(df['approx_cost(for two people)'], bins=12, kde=True, color='teal')
plt.title('Approximate Cost Distribution (for Two People)')
plt.xlabel('Cost')
plt.ylabel('Count')
plt.show()

In [ ]:
# Optional extra chart: Top 10 restaurants by votes
plt.figure(figsize=(12, 5))
sns.barplot(data=top_reviewed, x='name', y='votes', palette='magma')
plt.title('Top 10 Restaurants by Customer Votes')
plt.xlabel('Restaurant')
plt.ylabel('Votes')
plt.xticks(rotation=60, ha='right')
plt.show()

## 5) Conclusion

The dataset is dominated by **Dining** outlets, indicating that dine-in focused businesses are most common in this sample.
A larger share of restaurants do **not** support online ordering, and table booking is available in only a small fraction of outlets.
Average customer ratings are generally positive (around mid-to-high 3s), with **other** and **buffet** categories showing relatively stronger ratings.
Most restaurants fall in the **low-to-medium price range**, while premium-priced options are fewer.
Finally, a small group of outlets captures a disproportionately high number of customer votes, suggesting stronger brand visibility or customer loyalty.